In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, average_precision_score
from pycaret.classification import setup, compare_models, predict_model, pull
import h2o
from h2o.automl import H2OAutoML
from flaml import AutoML


In [12]:
df = pd.read_csv("../../data/creditcard.csv")

In [ ]:
# =========================
# COLUNAS
# =========================

X = df.drop(['Class'], axis=1)
y = df['Class'].astype(int)


In [ ]:
# =========================
# INSTALAR PYCARET
# =========================
#%pip install pycaret

for SEED in [42, 101, 223, 302, 404]:
    # =========================
    # DIVISÃO TREINO/TESTE
    # =========================
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=SEED, stratify=y
    )
    # DATAFRAME FINAL
    # =========================
    train_df = pd.DataFrame(X_train)
    train_df['Class'] = y_train.values

    test_df = pd.DataFrame(X_test)
    test_df['Class'] = y_test.values

    # =========================
    # PYCARET
    # =========================

    train_df = train_df.reset_index(drop=True)
    test_df = test_df.reset_index(drop=True)

    for fix_imbalance_method in ['editednearestneighborus', 'tomeklinks', 'SMOTE', 'SMOTEENN', 'SMOTETomek']:
        exp = setup(
            data=train_df,
            test_data=test_df,
            target="Class",
            index=False,  # PyCaret assigns a fresh index
            session_id=SEED,
            fix_imbalance=True,  # Handle class imbalance
            fix_imbalance_method=fix_imbalance_method,
            verbose=False
        )


        # Retorna os 10 melhores modelos treinados
        modelos_treinados = compare_models(n_select=10)

        # =========================
        # LEADERBOARD INTERNO DO PYCARET
        # =========================
        print(f"\n===== LEADERBOARD (Validação PyCaret) - Método de Balanceamento: {fix_imbalance_method} =====")
        print(pull())

        # =========================
        # MÉTRICAS DE TODOS OS MODELOS (No Test Set)
        # =========================
        resultados = []

        for model in modelos_treinados:
            model_name = model.__class__.__name__

            # Previsão no conjunto de teste
            preds_df = predict_model(model, data=test_df, verbose=False)

            y_true = test_df['Class'].values
            # No PyCaret 3.x, a coluna com o rótulo previsto é 'prediction_label'
            y_pred = preds_df['prediction_label'].astype(int).values

            # Métricas
            acc = round(accuracy_score(y_true, y_pred), 4)
            precision = round(precision_score(y_true, y_pred, pos_label=1, zero_division=0), 4)
            recall = round(recall_score(y_true, y_pred, pos_label=1, zero_division=0), 4)
            f1_weighted = round(f1_score(y_true, y_pred, pos_label=1, zero_division=0, average='weighted'), 4)
            f1 = round(f1_score(y_true, y_pred, pos_label=1, zero_division=0), 4)

            score_col = [col for col in preds_df.columns if 'score' in col]
            if score_col:
                pos_score_col = [c for c in score_col if '1' in str(c)]
                if pos_score_col:
                    y_score = preds_df[pos_score_col[0]].values
                else:
                    y_score = preds_df[score_col[0]].values
            else:
                y_score = y_pred

            average_precision = round(average_precision_score(y_true, y_score), 4)

            resultados.append({
                "Modelo": model_name,
                "Tipo": model_name,
                "Accuracy": acc,
                "Precision": precision,
                "Recall": recall,
                "F1-score": f1,
                "F1-score Weighted": f1_weighted,
                "average_precision": average_precision
            })

        # =========================
        # RESULTADOS
        # =========================
        df_resultados = pd.DataFrame(resultados)

        print("\n===== MELHORES MODELOS (Método de Balanceamento: {} SEED {}) =====".format(fix_imbalance_method, SEED))
        print(df_resultados.sort_values(by="average_precision", ascending=False))

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC,TT (Sec)
xgboost,Extreme Gradient Boosting,0.9996,0.9810,0.7865,0.9502,0.8585,0.8583,0.8632,39.9070
rf,Random Forest Classifier,0.9995,0.9500,0.7687,0.9345,0.8416,0.8414,0.8464,77.2270
et,Extra Trees Classifier,0.9995,0.9526,0.7687,0.9586,0.8505,0.8503,0.8568,47.5470
lda,Linear Discriminant Analysis,0.9994,0.9807,0.7587,0.8788,0.8129,0.8126,0.8155,40.1740
lr,Logistic Regression,0.9992,0.9603,0.6319,0.8747,0.7315,0.7311,0.7419,83.0620
ada,Ada Boost Classifier,0.9992,0.9681,0.7151,0.8154,0.7583,0.7579,0.7614,47.9720
dt,Decision Tree Classifier,0.9991,0.8816,0.7637,0.7503,0.7541,0.7537,0.7551,36.3960
ridge,Ridge Classifier,0.9989,0.9807,0.4163,0.8596,0.5598,0.5593,0.5971,43.1570
gbc,Gradient Boosting Classifier,0.9989,0.5919,0.4285,0.7967,0.5324,0.5319,0.5687,92.4030
knn,K Neighbors Classifier,0.9983,0.5896,0.0482,0.7917,0.0899,0.0897,0.1897,29.4190



===== LEADERBOARD (Validação PyCaret) - Método de Balanceamento: editednearestneighborus =====
                                    Model  Accuracy     AUC  Recall   Prec.  \
xgboost         Extreme Gradient Boosting    0.9996  0.9810  0.7865  0.9502   
rf               Random Forest Classifier    0.9995  0.9500  0.7687  0.9345   
et                 Extra Trees Classifier    0.9995  0.9526  0.7687  0.9586   
lda          Linear Discriminant Analysis    0.9994  0.9807  0.7587  0.8788   
lr                    Logistic Regression    0.9992  0.9603  0.6319  0.8747   
ada                  Ada Boost Classifier    0.9992  0.9681  0.7151  0.8154   
dt               Decision Tree Classifier    0.9991  0.8816  0.7637  0.7503   
ridge                    Ridge Classifier    0.9989  0.9807  0.4163  0.8596   
gbc          Gradient Boosting Classifier    0.9989  0.5919  0.4285  0.7967   
knn                K Neighbors Classifier    0.9983  0.5896  0.0482  0.7917   
dummy                    Dummy Clas

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC,TT (Sec)
xgboost,Extreme Gradient Boosting,0.9996,0.9784,0.7865,0.9509,0.8588,0.8586,0.8635,49.6410
rf,Random Forest Classifier,0.9995,0.9410,0.7713,0.9459,0.8481,0.8478,0.8531,81.2970
et,Extra Trees Classifier,0.9995,0.9513,0.7662,0.9610,0.8503,0.8500,0.8567,53.6470
lda,Linear Discriminant Analysis,0.9994,0.9808,0.7587,0.8788,0.8129,0.8126,0.8155,53.3640
lr,Logistic Regression,0.9992,0.9610,0.6546,0.8589,0.7415,0.7411,0.7487,52.6710
ada,Ada Boost Classifier,0.9992,0.9723,0.7023,0.7955,0.7430,0.7426,0.7455,67.8000
dt,Decision Tree Classifier,0.9991,0.8841,0.7687,0.7460,0.7541,0.7537,0.7553,44.4690
ridge,Ridge Classifier,0.9989,0.9808,0.4138,0.8591,0.5574,0.5569,0.5951,42.2080
gbc,Gradient Boosting Classifier,0.9987,0.5468,0.3794,0.6994,0.4607,0.4601,0.4960,127.1410
knn,K Neighbors Classifier,0.9983,0.5886,0.0482,0.8750,0.0902,0.0900,0.1985,56.6410



===== LEADERBOARD (Validação PyCaret) - Método de Balanceamento: tomeklinks =====
                                    Model  Accuracy     AUC  Recall   Prec.  \
xgboost         Extreme Gradient Boosting    0.9996  0.9784  0.7865  0.9509   
rf               Random Forest Classifier    0.9995  0.9410  0.7713  0.9459   
et                 Extra Trees Classifier    0.9995  0.9513  0.7662  0.9610   
lda          Linear Discriminant Analysis    0.9994  0.9808  0.7587  0.8788   
lr                    Logistic Regression    0.9992  0.9610  0.6546  0.8589   
ada                  Ada Boost Classifier    0.9992  0.9723  0.7023  0.7955   
dt               Decision Tree Classifier    0.9991  0.8841  0.7687  0.7460   
ridge                    Ridge Classifier    0.9989  0.9808  0.4138  0.8591   
gbc          Gradient Boosting Classifier    0.9987  0.5468  0.3794  0.6994   
knn                K Neighbors Classifier    0.9983  0.5886  0.0482  0.8750   
dummy                    Dummy Classifier    0.9

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC,TT (Sec)
et,Extra Trees Classifier,0.9996,0.9755,0.8220,0.9155,0.8641,0.8639,0.8662,8.5310
rf,Random Forest Classifier,0.9995,0.9660,0.8220,0.8919,0.8531,0.8529,0.8548,72.3950
xgboost,Extreme Gradient Boosting,0.9994,0.9763,0.8271,0.8566,0.8385,0.8382,0.8399,2.1270
lightgbm,Light Gradient Boosting Machine,0.9990,0.9711,0.8271,0.6709,0.7380,0.7375,0.7430,2.9960
dummy,Dummy Classifier,0.9983,0.5000,0.0000,0.0000,0.0000,0.0000,0.0000,0.3400
dt,Decision Tree Classifier,0.9977,0.8910,0.7838,0.4177,0.5441,0.5430,0.5707,8.1570
gbc,Gradient Boosting Classifier,0.9938,0.9757,0.8651,0.2002,0.3248,0.3229,0.4141,147.2430
nb,Naive Bayes,0.9922,0.9653,0.7510,0.1500,0.2499,0.2477,0.3333,0.3360
lr,Logistic Regression,0.9891,0.9804,0.8880,0.1257,0.2200,0.2176,0.3314,9.4570
ridge,Ridge Classifier,0.9875,0.9731,0.8222,0.1042,0.1849,0.1824,0.2900,0.3280



===== LEADERBOARD (Validação PyCaret) - Método de Balanceamento: SMOTE =====
                                    Model  Accuracy     AUC  Recall   Prec.  \
et                 Extra Trees Classifier    0.9996  0.9755  0.8220  0.9155   
rf               Random Forest Classifier    0.9995  0.9660  0.8220  0.8919   
xgboost         Extreme Gradient Boosting    0.9994  0.9763  0.8271  0.8566   
lightgbm  Light Gradient Boosting Machine    0.9990  0.9711  0.8271  0.6709   
dummy                    Dummy Classifier    0.9983  0.5000  0.0000  0.0000   
dt               Decision Tree Classifier    0.9977  0.8910  0.7838  0.4177   
gbc          Gradient Boosting Classifier    0.9938  0.9757  0.8651  0.2002   
nb                            Naive Bayes    0.9922  0.9653  0.7510  0.1500   
lr                    Logistic Regression    0.9891  0.9804  0.8880  0.1257   
ridge                    Ridge Classifier    0.9875  0.9731  0.8222  0.1042   
ada                  Ada Boost Classifier    0.9875  

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC,TT (Sec)
rf,Random Forest Classifier,0.9995,0.9734,0.8246,0.8660,0.8432,0.8429,0.8440,170.8700
et,Extra Trees Classifier,0.9995,0.9780,0.8271,0.8858,0.8537,0.8535,0.8549,128.5720
xgboost,Extreme Gradient Boosting,0.9994,0.9798,0.8321,0.8236,0.8254,0.8251,0.8263,125.4430
lightgbm,Light Gradient Boosting Machine,0.9990,0.9653,0.8221,0.6740,0.7380,0.7375,0.7425,125.2340
dt,Decision Tree Classifier,0.9978,0.8911,0.7840,0.4250,0.5501,0.5491,0.5757,126.7790
gbc,Gradient Boosting Classifier,0.9939,0.9732,0.8651,0.2039,0.3297,0.3278,0.4180,207.6660
nb,Naive Bayes,0.9924,0.9654,0.7485,0.1540,0.2553,0.2531,0.3372,130.1890
lr,Logistic Regression,0.9893,0.9800,0.8854,0.1284,0.2240,0.2216,0.3345,216.5160
ada,Ada Boost Classifier,0.9884,0.9778,0.8754,0.1183,0.2082,0.2058,0.3190,140.7310
ridge,Ridge Classifier,0.9869,0.9727,0.8223,0.1001,0.1784,0.1758,0.2841,123.5860



===== LEADERBOARD (Validação PyCaret) - Método de Balanceamento: SMOTEENN =====
                                    Model  Accuracy     AUC  Recall   Prec.  \
rf               Random Forest Classifier    0.9995  0.9734  0.8246  0.8660   
et                 Extra Trees Classifier    0.9995  0.9780  0.8271  0.8858   
xgboost         Extreme Gradient Boosting    0.9994  0.9798  0.8321  0.8236   
lightgbm  Light Gradient Boosting Machine    0.9990  0.9653  0.8221  0.6740   
dt               Decision Tree Classifier    0.9978  0.8911  0.7840  0.4250   
gbc          Gradient Boosting Classifier    0.9939  0.9732  0.8651  0.2039   
nb                            Naive Bayes    0.9924  0.9654  0.7485  0.1540   
lr                    Logistic Regression    0.9893  0.9800  0.8854  0.1284   
ada                  Ada Boost Classifier    0.9884  0.9778  0.8754  0.1183   
ridge                    Ridge Classifier    0.9869  0.9727  0.8223  0.1001   
lda          Linear Discriminant Analysis    0.986

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC,TT (Sec)
et,Extra Trees Classifier,0.9996,0.9726,0.8246,0.9183,0.8670,0.8668,0.8690,128.3560
rf,Random Forest Classifier,0.9995,0.9710,0.8245,0.8854,0.8523,0.8520,0.8534,175.8860
xgboost,Extreme Gradient Boosting,0.9995,0.9816,0.8245,0.8583,0.8388,0.8385,0.8398,123.9030
lightgbm,Light Gradient Boosting Machine,0.9990,0.9595,0.8144,0.6748,0.7351,0.7346,0.7393,124.7280
dummy,Dummy Classifier,0.9983,0.5000,0.0000,0.0000,0.0000,0.0000,0.0000,132.0100
dt,Decision Tree Classifier,0.9979,0.8899,0.7815,0.4472,0.5679,0.5670,0.5898,129.5200
gbc,Gradient Boosting Classifier,0.9937,0.9748,0.8676,0.1985,0.3226,0.3207,0.4128,217.4520
nb,Naive Bayes,0.9922,0.9653,0.7510,0.1508,0.2511,0.2489,0.3343,121.6360
lr,Logistic Regression,0.9883,0.9795,0.8829,0.1182,0.2084,0.2060,0.3205,127.1790
ada,Ada Boost Classifier,0.9879,0.9705,0.8701,0.1125,0.1991,0.1967,0.3102,142.5610



===== LEADERBOARD (Validação PyCaret) - Método de Balanceamento: SMOTETomek =====
                                    Model  Accuracy     AUC  Recall   Prec.  \
et                 Extra Trees Classifier    0.9996  0.9726  0.8246  0.9183   
rf               Random Forest Classifier    0.9995  0.9710  0.8245  0.8854   
xgboost         Extreme Gradient Boosting    0.9995  0.9816  0.8245  0.8583   
lightgbm  Light Gradient Boosting Machine    0.9990  0.9595  0.8144  0.6748   
dummy                    Dummy Classifier    0.9983  0.5000  0.0000  0.0000   
dt               Decision Tree Classifier    0.9979  0.8899  0.7815  0.4472   
gbc          Gradient Boosting Classifier    0.9937  0.9748  0.8676  0.1985   
nb                            Naive Bayes    0.9922  0.9653  0.7510  0.1508   
lr                    Logistic Regression    0.9883  0.9795  0.8829  0.1182   
ada                  Ada Boost Classifier    0.9879  0.9705  0.8701  0.1125   
ridge                    Ridge Classifier    0.9

In [ ]:
# =========================
# INSTALAR H2O
# =========================
# %pip install h2o

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, average_precision_score
import h2o
from h2o.automl import H2OAutoML

# =========================
# INICIAR H2O
# =========================
h2o.init()

target_col = 'Class' # Nome da coluna alvo para evitar conflito com a variável y do split

for SEED in [42, 101, 223, 302, 404]:
    
    # =========================
    # DIVISÃO TREINO/TESTE
    # =========================
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=SEED, stratify=y
    )
    
    # =========================
    # DATAFRAME FINAL
    # =========================
    train_df = pd.DataFrame(X_train)
    train_df[target_col] = y_train.values

    test_df = pd.DataFrame(X_test)
    test_df[target_col] = y_test.values

    # =========================
    # PARA H2O
    # =========================
    train_h2o = h2o.H2OFrame(train_df)
    test_h2o = h2o.H2OFrame(test_df)

    train_h2o[target_col] = train_h2o[target_col].asfactor()
    test_h2o[target_col] = test_h2o[target_col].asfactor()

    x_features = train_h2o.columns[:-1]

    # =========================
    # AUTO ML
    # =========================
    aml = H2OAutoML(
        max_models=20,
        seed=SEED,
        balance_classes=True,
        sort_metric="f1"
    )

    aml.train(x=x_features, y=target_col, training_frame=train_h2o)

    # =========================
    # LEADERBOARD
    # =========================
    print("\n===== LEADERBOARD =====")
    print(aml.leaderboard)

    # =========================
    # MÉTRICAS DE TODOS OS MODELOS
    # =========================
    leaderboard = aml.leaderboard.as_data_frame()
    resultados = []

    for model_id in leaderboard['model_id']:
        model = h2o.get_model(model_id)

        preds = model.predict(test_h2o).as_data_frame()

        y_true = test_df[target_col].values
        y_pred = preds['predict'].astype(int).values
        y_score = preds['p1'].values

        # Métricas
        acc = accuracy_score(y_true, y_pred)
        precision = precision_score(y_true, y_pred, pos_label=1, zero_division=0)
        recall = recall_score(y_true, y_pred, pos_label=1, zero_division=0)
        f1_weighted = f1_score(y_true, y_pred, pos_label=1, zero_division=0, average='weighted')
        f1 = f1_score(y_true, y_pred, pos_label=1, zero_division=0)
        average_precision = average_precision_score(y_true, y_score)

        resultados.append({
            "Seed": SEED,
            "Modelo": model_id,
            "Tipo": model.algo,
            "Accuracy": round(acc, 4),
            "Precision": round(precision, 4),
            "Recall": round(recall, 4),
            "F1-score": round(f1, 4),
            "F1-score Weighted": round(f1_weighted, 4),
            "average_precision": round(average_precision, 4)
        })

    # =========================
    # RESULTADO PARCIAL DA SEED
    # =========================
    df_resultados = pd.DataFrame(resultados)
    print(f"\n===== MELHORES MODELOS (SEED {SEED}) =====")
    print(df_resultados.sort_values(by="average_precision", ascending=False).head())

    h2o.remove_all()
    h2o.init()


[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: /opt/homebrew/opt/python@3.11/bin/python3.11 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
Checking whether there is an H2O instance running at http://localhost:54321. connected.


H2O_cluster_uptime:,24 days 20 hours 21 mins
H2O_cluster_timezone:,America/Sao_Paulo
H2O_data_parsing_timezone:,UTC
H2O_cluster_version:,3.46.0.12
H2O_cluster_version_age:,2 months and 19 days
H2O_cluster_name:,H2O_from_python_lannaces_gfr9u3
H2O_cluster_total_nodes:,1
H2O_cluster_free_memory:,458.9 Mb
H2O_cluster_total_cores:,10
H2O_cluster_allowed_cores:,10
H2O_cluster_status:,"locked, healthy"



+------------------------------------------------------------------+
| You are running the community edition of H2O-3 OSS.              |
|                                                                  |
| For commercial use, H2O-3 Secure is now recommended.             |
| This includes production support, CVE fixes, multi-node scaling, |
| model artifact extraction, and more.                             |
| See h2o.ai/h2o-3/oss-vs-secure for additional details.           |
| Contact enterprise@h2o.ai to upgrade.                            |
+------------------------------------------------------------------+
Parse progress: |████████████████████████████████████████████████████████████████| (done) 100%
Parse progress: |████████████████████████████████████████████████████████████████| (done) 100%
AutoML progress: |
18:31:20.228: AutoML: XGBoost is not available; skipping it.

███████████████████████████████████████████████████████████████| (done) 100%

===== LEADERBOARD =====
model

In [ ]:
# =========================
# INSTALAR FLAML
# =========================
%pip install "flaml[automl]"

for SEED in [42, 101, 223, 302, 404]:
    # =========================
    # DIVISÃO TREINO/TESTE
    # =========================
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=SEED, stratify=y
    )
    # =========================
    # TREINAMENTO COM FLAML
    # =========================
    automl = AutoML()
    settings = {
        "task": "classification",
        "metric": "f1", # Métrica orientada a custo/desbalanceamento
        "time_budget": 300,
        "config_list": [
            {
                "model": "xgboost",
                "scale_pos_weight": 10, # Ajuste o peso conforme o grau de desbalanceamento
            },
            {
                "model": "lgbm",
                "class_weight": "balanced",
            }
        ],
        "log_file_name": "flaml_cost_sensitive.log",
    }
    automl.fit(X_train=X_train, y_train=y_train, **settings)

    # =========================
    # RESULTADOS DO MELHOR MODELO
    # =========================
    print("\n===== MELHOR RESULTADO FLAML =====")
    print("Melhor modelo:", automl.best_estimator)
    print("Melhores parâmetros:", automl.best_config)
    print("Melhor loss:", automl.best_loss)

    # =========================
    # SELECIONAR OS 5 MELHORES ESTIMADORES
    # =========================
    perdas_por_estimador = automl.best_loss_per_estimator
    melhores_estimadores = sorted(
        perdas_por_estimador.items(),
        key=lambda item: item[1]
    )[:5]

    if not melhores_estimadores:
        raise RuntimeError("O FLAML não encontrou estimadores avaliados.")

    print("\n===== 5 MELHORES ESTIMADORES (VALIDAÇÃO FLAML) =====")
    print(pd.DataFrame(
        melhores_estimadores,
        columns=["Modelo", "Loss validação"]
    ).to_string(index=False))

    # =========================
    # MÉTRICAS DOS 5 MELHORES NO TESTE
    # =========================
    resultados = []
    y_true = y_test.to_numpy()

    for rank, (nome_modelo, loss_validacao) in enumerate(melhores_estimadores, start=1):
        modelo = automl.best_model_for_estimator(nome_modelo)
        y_pred = modelo.predict(X_test)

        if hasattr(modelo, "predict_proba"):
            probabilidades = modelo.predict_proba(X_test)
            classes = list(modelo.classes_)
            y_score = probabilidades[:, classes.index(1)]
        elif hasattr(modelo, "decision_function"):
            y_score = modelo.decision_function(X_test)
            if list(modelo.classes_).index(1) == 0:
                y_score = -y_score
        else:
            y_score = y_pred

        resultados.append({
            "Posição": rank,
            "Modelo": nome_modelo,
            "Loss validação": round(float(loss_validacao), 4),
            "Accuracy": round(accuracy_score(y_true, y_pred), 4),
            "Precision": round(precision_score(y_true, y_pred, pos_label=1, zero_division=0), 4),
            "Recall": round(recall_score(y_true, y_pred, pos_label=1, zero_division=0), 4),
            "F1-score Weighted": round(f1_score(y_true, y_pred, average="weighted", zero_division=0), 4),
            "F1-score": round(f1_score(y_true, y_pred, pos_label=1, zero_division=0), 4),
            "Average Precision": round(average_precision_score(y_true, y_score), 4),
        })

    df_resultados_flaml = pd.DataFrame(resultados)
    print("\n===== MÉTRICAS DOS 5 MELHORES MODELOS FLAML (TESTE) =====")
    print(df_resultados_flaml.to_string(index=False))
